# Employee Attrition — Machine Learning Model Development

This notebook covers Phase 3: Model Development. We will train and compare multiple machine learning algorithms to predict employee attrition.


## 1. Data Loading & Preparation

We start by loading the preprocessed, ML-ready dataset (`attrition_ml_ready.csv`) and splitting it into training and testing sets. We use stratified sampling to handle class imbalance.


In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')


In [2]:
# Load the ML-ready dataset
data_path = '../data/attrition_ml_ready.csv'
df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape}")
df.head()


Dataset shape: (1470, 51)


,Age,Attrition,DailyRate,DistanceFromHome,Education,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,...,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Divorced,MaritalStatus_Married,MaritalStatus_Single
0,0.446350,1,0.742527,-1.010909,-0.891688,-1.701283,-0.660531,-1.224745,1.383138,0.379672,...,False,False,False,False,False,True,False,False,False,True
1,1.322365,0,-1.297775,-0.147150,-1.868426,-1.699621,0.254625,0.816497,-0.240677,-1.026167,...,False,False,False,False,True,False,False,False,True,False
2,0.008343,1,1.414363,-0.887515,-0.891688,-1.696298,1.169781,0.816497,1.284725,-1.026167,...,True,False,False,False,False,False,False,False,False,True
3,-0.429664,0,1.461466,-0.764121,1.061787,-1.694636,1.169781,-1.224745,-0.486709,0.379672,...,False,False,False,False,True,False,False,False,True,False
4,-1.086676,0,-0.524295,-0.887515,-1.868426,-1.691313,-1.575686,0.816497,-1.274014,0.379672,...,True,False,False,False,False,False,False,False,True,False


In [3]:
# Separate features (X) and target (y)
target_col = 'Attrition'
X = df.drop(columns=[target_col])
y = df[target_col]

# Split into training (80%) and testing (20%) sets
# stratify=y ensures the same proportion of Attrition instances in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: X={X_train.shape}, y={y_train.shape}")
print(f"Testing set : X={X_test.shape}, y={y_test.shape}")
print("\nTarget distribution in Training set:\n", y_train.value_counts(normalize=True))
print("\nTarget distribution in Testing set:\n", y_test.value_counts(normalize=True))


Training set: X=(1176, 50), y=(1176,)
Testing set : X=(294, 50), y=(294,)

Target distribution in Training set:
 Attrition
0    0.838435
1    0.161565
Name: proportion, dtype: float64

Target distribution in Testing set:
 Attrition
0    0.840136
1    0.159864
Name: proportion, dtype: float64


## 2. Model Training

We will train three classification models: Logistic Regression, Decision Tree, and Random Forest.


In [4]:
# 2.1 Logistic Regression
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
print("Logistic Regression model trained.")


Logistic Regression model trained.


In [5]:
# 2.2 Decision Tree Classifier
dt_model = DecisionTreeClassifier(random_state=42, max_depth=5)
dt_model.fit(X_train, y_train)
print("Decision Tree model trained.")


Decision Tree model trained.


In [6]:
# 2.3 Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
rf_model.fit(X_train, y_train)
print("Random Forest model trained.")


Random Forest model trained.


## 3. Model Evaluation & Comparison

We will evaluate the models using Accuracy, Precision, Recall, and F1-Score. In attrition prediction, **Recall** (identifying as many at-risk employees as possible) and **F1-Score** (balance between precision and recall) are crucial due to class imbalance.


In [7]:
def evaluate_model(model, model_name, X_test, y_test):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    return {
        'Model': model_name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4)
    }

# Evaluate all models
results = []
models = {
    'Logistic Regression': lr_model,
    'Decision Tree': dt_model,
    'Random Forest': rf_model
}

for name, model in models.items():
    res = evaluate_model(model, name, X_test, y_test)
    results.append(res)

# Create a comparison dataframe
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='F1-Score', ascending=False).reset_index(drop=True)

print("=== Model Comparison Report ===")
display(results_df)


=== Model Comparison Report ===


,Model,Accuracy,Precision,Recall,F1-Score
0,Logistic Regression,0.8605,0.6154,0.3404,0.4384
1,Decision Tree,0.8435,0.5294,0.1915,0.2812
2,Random Forest,0.8367,0.4444,0.0851,0.1429


In [8]:
# Detailed Classification Report for the top model
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]

print(f"\nDetailed Classification Report for: {best_model_name}")
y_pred_best = best_model.predict(X_test)
print(classification_report(y_test, y_pred_best))



Detailed Classification Report for: Logistic Regression
              precision    recall  f1-score   support

           0       0.88      0.96      0.92       247
           1       0.62      0.34      0.44        47

    accuracy                           0.86       294
   macro avg       0.75      0.65      0.68       294
weighted avg       0.84      0.86      0.84       294



## 4. Model Selection & Export

We select the best model based on F1-Score and export it for future use.


In [9]:
# Select best model and save
os.makedirs('../models', exist_ok=True)
model_filepath = f'../models/best_{best_model_name.replace(" ", "_").lower()}_model.pkl'

joblib.dump(best_model, model_filepath)
print(f"Successfully saved the {best_model_name} model to {model_filepath}")


Successfully saved the Logistic Regression model to ../models/best_logistic_regression_model.pkl
